# Lab 43: Tracking annotator drift

Extend [Lab 40](../40-annotation-quality/): track each annotator's agreement-with-consensus over multiple rounds to catch a rater whose labels degrade. Annotator drift masquerades as model drift — only per-annotator tracking tells them apart. Fill in the `TODO` cells; reference in `solution/`.

## Step 0: Setup

In [ ]:
import json
import pathlib
from sklearn.metrics import cohen_kappa_score
print("tracking annotator agreement-with-consensus over rounds")

## Step 1: Multiple annotation rounds

In [ ]:
# Lab 40 measured agreement at one point in time. But annotators drift too: a rater's
# internal rubric slips, and the labels you trust quietly degrade. Here are three rounds
# of annotation (the same three annotators, fresh items each round), binary "correct".
with open("./annotation_rounds.jsonl") as f:
    rounds = [json.loads(line) for line in f]
by_round={r:[x for x in rounds if x["round"]==r] for r in sorted({x["round"] for x in rounds})}
print(f"{len(rounds)} items across rounds {list(by_round)}")
def consensus(rows): return [1 if (r["a1"]+r["a2"]+r["a3"])>=2 else 0 for r in rows]

## Step 2: Per-round agreement-with-consensus

In [ ]:
ann=["a1","a2","a3"]
trend={a:[] for a in ann}
# TODO: for each round, compute consensus, then cohen_kappa_score(annotator, consensus)
# for each annotator; append to trend[a] and print the per-round line.
raise NotImplementedError

## Step 3: Detect the drift

Watch the trajectory, not a single round.

In [ ]:
DROP_TOL=0.25
# TODO: for each annotator, compute the drop from round 1 to the last round and flag any
# annotator whose agreement-with-consensus fell by more than DROP_TOL.
raise NotImplementedError

## Step 4: React — down-weight and re-ceiling

In [ ]:
# Two consequences of a drifting annotator:
# 1) Down-weight them in consensus until re-calibrated (reliability-weighted vote).
weights={a:max(trend[a][-1],0.0) for a in ann}   # weight by latest-round agreement
print("reliability weights (latest round):", weights)
def weighted_consensus(row, w):
    score=sum(w[a]*(1 if row[a]==1 else -1) for a in ann)
    return 1 if score>=0 else 0
# 2) The drift moves the CEILING: inter-annotator agreement (Lab 40) is only meaningful
# among calibrated annotators. Recompute it excluding (or after fixing) the drifter.
latest=by_round[max(by_round)]
from statistics import mean
pair_keep=mean([cohen_kappa_score([r["a1"] for r in latest],[r["a2"] for r in latest])])
print(f"latest-round a1-a2 agreement (the calibrated pair): {pair_keep:.2f}")
print("Use the calibrated pair for the ceiling; the drifter's labels are noise until fixed.")

## Step 5: Why it matters for the judge

In [ ]:
# Why this matters for the judge (Lab 40 -> Lab 38): the judge is validated against the
# human consensus and bounded by the human ceiling. If an annotator drifts and you do not
# notice, BOTH move - the consensus shifts toward noise and the ceiling looks lower - and
# you will wrongly conclude the judge got worse. Track annotators over time so you can tell
# annotator drift apart from model drift.
print("Model drift and annotator drift look the same in a single judge-vs-consensus number.")
print("Only per-annotator tracking over rounds tells them apart.")

## Step 6: The discipline

In [ ]:
# The discipline: validation is not one-and-done. Re-measure agreement every round, watch
# each annotator's trajectory, down-weight or re-train a drifter, and recompute the ceiling
# among the calibrated raters before you trust any judged metric again.
print("Watch the annotators, not just the model. A drifting rater corrupts the ground")
print("truth everything downstream is measured against.")

## What you built

A per-annotator, per-round tracker of agreement-with-consensus (Cohen's κ) that flags a drifting annotator by trajectory, a reliability-weighted consensus that down-weights the drifter, and the link back to the judge: annotator drift and model drift are indistinguishable in a single judge-vs-consensus number, so only tracking annotators over time separates them.

**Where this simplifies:** three rounds of 12 items is a teaching size — real drift is noisier and needs more rounds and items to call confidently (watch the trend, and use a tolerance band, not a single-round dip); the reliability weighting is a simple linear scheme (consider adjudication or a held-out gold set instead); and a flagged annotator needs re-calibration (shared examples, guideline review), not just removal.

This closes the evaluation-quality thread: [Lab 40](../40-annotation-quality/) gave you the ceiling; this keeps the ceiling current as the annotators themselves change.